# 04 - Encoder-decoder architectures, teacher forcing and training

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
%matplotlib inline
import torch
from src import config as C
from src.seq2seq import build_model
from src.utils import count_parameters
from src.vocabulary import Vocabulary
src_v = Vocabulary.load(C.SRC_VOCAB_FILE); tgt_v = Vocabulary.load(C.TGT_VOCAB_FILE)

## Architectures
**E1 baseline**: Embedding(128) → LSTM encoder(256) → (h_S, c_S) initialises → LSTM decoder(256) → Linear → softmax.
The whole input is squeezed into one fixed-size state: long or information-rich inputs get compressed.

**E2 attention**: same encoder; at every decoder step Bahdanau attention computes
e(t,i) = vᵀ tanh(W_h h_i + W_s s_{t−1}), α(t,i) = softmax_i e(t,i), c(t) = Σ_i α(t,i) h_i;
the LSTM input is [emb(y_{t−1}); c(t)] and the output layer sees [s_t; c(t); emb(y_{t−1})].

In [ ]:
for kind in ['baseline', 'attention']:
    cfg = C.ModelConfig(model_type=kind)
    m = build_model(cfg, len(src_v), len(tgt_v))
    print(f'{kind:<10} {count_parameters(m):>12,} parameters'); print(m, '\n')

## Tensor shapes through one forward pass

In [ ]:
from src.dataset import EventDataset, collate
train = pd.read_csv(C.SPLIT_FILES['train']).head(8)
ds = EventDataset(train, src_v, tgt_v, 40, 60); src, src_len, tgt, _ = collate([ds[i] for i in range(8)])
m = build_model(C.ModelConfig(model_type='attention'), len(src_v), len(tgt_v)).eval()
enc_out, (h, c) = m.encoder(src, src_len)
print('src', tuple(src.shape), '| encoder outputs', tuple(enc_out.shape), '| h', tuple(h.shape), '| c', tuple(c.shape))
logits, attn = m(src, src_len, tgt, teacher_forcing=1.0, return_attention=True)
print('target', tuple(tgt.shape), '-> logits', tuple(logits.shape), '| attention', tuple(attn.shape))
print('attention rows sum to 1:', torch.allclose(attn.sum(-1), torch.ones_like(attn.sum(-1))))

## Teacher forcing
At step t the next decoder input is the **ground-truth** y_t with probability `teacher_forcing`
(0.5 here), otherwise the model's own prediction. With 1.0 training is fast and stable but the model never
sees its own mistakes (**exposure bias**); with 0.0 it trains exactly as it will generate but learns
slowly at first. At inference there is no reference, so generated tokens are always fed back.

## Loss and optimiser
Cross-entropy per non-pad token: L = −(1/N) Σ log P(y_t | y_<t, X) (`ignore_index=<pad>`). Adam (lr 1e-3)
updates each parameter with adaptive, momentum-smoothed steps from the gradients obtained by
backpropagation through time; gradients are clipped to norm 1.0; LR halved when the validation loss
plateaus; early stopping after 4 epochs without improvement; best checkpoint saved.

## Training
Training is done by `scripts/03_train.py` (or `00_colab_run_all.ipynb` on GPU). Set `TRAIN = True` to
retrain from here.

In [ ]:
TRAIN = False
if TRAIN:
    from src.config import ExperimentConfig
    from src.training import train_model
    for kind in ['baseline', 'attention']:
        cfg = ExperimentConfig(); cfg.model.model_type = kind
        print(train_model(cfg))

In [ ]:
from src.visualization import plot_learning_curves
hist = {d.name: pd.read_csv(d / 'history.csv') for d in sorted(C.MODELS_DIR.iterdir()) if (d / 'history.csv').exists()}
for k, h in hist.items(): print(k); display(h.round(3))
if hist: plot_learning_curves(hist); plt.show()

In [ ]:
summ = {d.name: json.load(open(d / 'train_summary.json')) for d in sorted(C.MODELS_DIR.iterdir()) if (d / 'train_summary.json').exists()}
pd.DataFrame(summ).T[['n_params','best_epoch','epochs_run','best_val_loss','best_val_ppl','train_time_s','device','train_size']]

**How to read the curves.** Train and validation losses both decreasing = still learning (more epochs
could help). Validation loss rising while training loss keeps falling = overfitting: early stopping keeps the
best epoch. Both high and flat = underfitting (capacity or learning-rate problem). Note that the training loss
is measured with teacher forcing 0.5 and the validation loss with 1.0, so they are not on exactly the same footing.